In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

#Load and preprocess the data:

data_path = "C:/Users/olufe/projects/Journal/dataset/HomeB_unsupervise_sim_combined_shuffled.csv"
data = pd.read_csv(data_path)

# Assuming the target column is named "Label", you can adjust the name accordingly
X = data.drop(columns=["Label"])
y = data["Label"]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [3]:
#Build the Deep Autoencoder:
def build_autoencoder(input_dim):
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(64, activation='relu')(input_layer)
    encoded = Dense(32, activation='relu')(encoded)
    encoded = Dense(16, activation='relu')(encoded)
    decoded = Dense(32, activation='relu')(encoded)
    decoded = Dense(64, activation='relu')(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)

    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')
    return autoencoder

# Assuming the input dimension is the number of features in the dataset
input_dim = X_train_scaled.shape[1]
autoencoder = build_autoencoder(input_dim)


In [4]:
#Train the autoencoder:
# Define early stopping to avoid overfitting
early_stopping = EarlyStopping(patience=3, restore_best_weights=True)

# Train the autoencoder
autoencoder.fit(X_train_scaled, X_train_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=[early_stopping])


Epoch 1/50
398/398 [==============================] - 3s 5ms/step - loss: 0.6999 - val_loss: 0.5885
Epoch 2/50
398/398 [==============================] - 2s 6ms/step - loss: 0.5421 - val_loss: 0.5136
Epoch 3/50
398/398 [==============================] - 2s 6ms/step - loss: 0.4892 - val_loss: 0.4779
Epoch 4/50
398/398 [==============================] - 2s 5ms/step - loss: 0.4625 - val_loss: 0.4573
Epoch 5/50
398/398 [==============================] - 2s 4ms/step - loss: 0.4448 - val_loss: 0.4444
Epoch 6/50
398/398 [==============================] - 2s 5ms/step - loss: 0.4312 - val_loss: 0.4328
Epoch 7/50
398/398 [==============================] - 2s 5ms/step - loss: 0.4215 - val_loss: 0.4247
Epoch 8/50
398/398 [==============================] - 2s 4ms/step - loss: 0.4133 - val_loss: 0.4170
Epoch 9/50
398/398 [==============================] - 2s 5ms/step - loss: 0.4073 - val_loss: 0.4159
Epoch 10/50
398/398 [==============================] - 2s 5ms/step - loss: 0.4024 - val_loss: 0.4094

In [4]:
#Evaluate the autoencoder:

# Obtain the reconstruction error on the test set
X_test_reconstructed = autoencoder.predict(X_test_scaled)
mse = np.mean(np.square(X_test_scaled - X_test_reconstructed), axis=1)

# Set a threshold to classify anomalies
threshold = np.percentile(mse, 95)
y_pred = [1 if err > threshold else 0 for err in mse]

# Evaluate performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
tnr = tn / (tn + fp)
fpr = fp / (tn + fp)
fnr = fn / (fn + tp)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, mse)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("TNR:", tnr)
print("FPR:", fpr)
print("FNR:", fnr)
print("F1-score:", f1)
print("AUC:", auc)

221/221 [==============================] - 1s 3ms/step
Accuracy: 0.9500424568355506
Precision: 0.0423728813559322
Recall: 0.5172413793103449
TNR: 0.9518260622424328
FPR: 0.04817393775756715
FNR: 0.4827586206896552
F1-score: 0.0783289817232376
AUC: 0.8591435417718168


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

#Load and preprocess the data:

data_path = "C:/Users/olufe/projects/Journal/dataset/HomeB_unsupervise_real_combined_shuffled.csv"
data = pd.read_csv(data_path)

# Assuming the target column is named "Label", you can adjust the name accordingly
X = data.drop(columns=["Label"])
y = data["Label"]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [6]:
#Build the Deep Autoencoder:
def build_autoencoder(input_dim):
    input_layer = Input(shape=(input_dim,))
    encoded = Dense(64, activation='relu')(input_layer)
    encoded = Dense(32, activation='relu')(encoded)
    encoded = Dense(16, activation='relu')(encoded)
    decoded = Dense(32, activation='relu')(encoded)
    decoded = Dense(64, activation='relu')(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)

    autoencoder = Model(input_layer, decoded)
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')
    return autoencoder

# Assuming the input dimension is the number of features in the dataset
input_dim = X_train_scaled.shape[1]
autoencoder = build_autoencoder(input_dim)

In [7]:
#Train the autoencoder:
# Define early stopping to avoid overfitting
early_stopping = EarlyStopping(patience=3, restore_best_weights=True)

# Train the autoencoder
autoencoder.fit(X_train_scaled, X_train_scaled, epochs=50, batch_size=64, validation_split=0.1, callbacks=[early_stopping])

Epoch 1/50
592/592 [==============================] - 3s 4ms/step - loss: 0.6730 - val_loss: 0.5633
Epoch 2/50
592/592 [==============================] - 3s 4ms/step - loss: 0.5365 - val_loss: 0.5008
Epoch 3/50
592/592 [==============================] - 2s 4ms/step - loss: 0.4907 - val_loss: 0.4701
Epoch 4/50
592/592 [==============================] - 2s 4ms/step - loss: 0.4683 - val_loss: 0.4551
Epoch 5/50
592/592 [==============================] - 3s 4ms/step - loss: 0.4555 - val_loss: 0.4456
Epoch 6/50
592/592 [==============================] - 3s 4ms/step - loss: 0.4469 - val_loss: 0.4370
Epoch 7/50
592/592 [==============================] - 3s 4ms/step - loss: 0.4399 - val_loss: 0.4335
Epoch 8/50
592/592 [==============================] - 2s 4ms/step - loss: 0.4344 - val_loss: 0.4281
Epoch 9/50
592/592 [==============================] - 3s 4ms/step - loss: 0.4290 - val_loss: 0.4229
Epoch 10/50
592/592 [==============================] - 3s 5ms/step - loss: 0.4234 - val_loss: 0.4177

In [8]:
#Evaluate the autoencoder:

# Obtain the reconstruction error on the test set
X_test_reconstructed = autoencoder.predict(X_test_scaled)
mse = np.mean(np.square(X_test_scaled - X_test_reconstructed), axis=1)

# Set a threshold to classify anomalies
threshold = np.percentile(mse, 95)
y_pred = [1 if err > threshold else 0 for err in mse]

# Evaluate performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
tnr = tn / (tn + fp)
fpr = fp / (tn + fp)
fnr = fn / (fn + tp)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, mse)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("TNR:", tnr)
print("FPR:", fpr)
print("FNR:", fnr)
print("F1-score:", f1)
print("AUC:", auc)

329/329 [==============================] - 1s 3ms/step
Accuracy: 0.7903348554033486
Precision: 0.0741444866920152
Recall: 0.022209567198177675
TNR: 0.9443809958885335
FPR: 0.055619004111466426
FNR: 0.9777904328018223
F1-score: 0.034180543382997364
AUC: 0.6431098727224858
